In [1]:
from pathlib import Path
import pandas as pd

path = Path("../data/raw/binance/BTC_USDT-5m.feather")

df = pd.read_feather(path)

print(df.head())
print()
print(df.info())

                       date      open      high       low     close     volume
0 2023-01-01 00:00:00+00:00  16541.77  16544.76  16527.51  16535.38  486.60903
1 2023-01-01 00:05:00+00:00  16534.91  16540.43  16522.55  16526.67  391.19043
2 2023-01-01 00:10:00+00:00  16526.67  16530.87  16520.00  16520.69  294.73889
3 2023-01-01 00:15:00+00:00  16521.26  16537.73  16517.72  16534.94  481.18777
4 2023-01-01 00:20:00+00:00  16534.94  16540.66  16532.33  16535.54  309.53189

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236438 entries, 0 to 236437
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype              
---  ------  --------------   -----              
 0   date    236438 non-null  datetime64[ns, UTC]
 1   open    236438 non-null  float64            
 2   high    236438 non-null  float64            
 3   low     236438 non-null  float64            
 4   close   236438 non-null  float64            
 5   volume  236438 non-null  float64            
dtypes: datetim

In [2]:
print(df.columns.tolist())
print()
print("shape:", df.shape)

['date', 'open', 'high', 'low', 'close', 'volume']

shape: (236438, 6)


# Sanity Check

In [3]:
print("start:", df["date"].min())
print("end:  ", df["date"].max())

print("sorted:", df["date"].is_monotonic_increasing)
print("duplicate timestamps:", df["date"].duplicated().sum())

diff = df["date"].diff()

print("\nMost common intervals:")
print(diff.value_counts().head())

print("\nNon-5m intervals:", (diff.dropna() != pd.Timedelta(minutes=5)).sum())

start: 2023-01-01 00:00:00+00:00
end:   2025-04-01 05:35:00+00:00
sorted: True
duplicate timestamps: 0

Most common intervals:
date
0 days 00:05:00    236435
0 days 01:25:00         1
0 days 05:15:00         1
Name: count, dtype: int64

Non-5m intervals: 2


## OHLC sanity check

In [4]:
bad_high = (
    (df["high"] < df["open"]) |
    (df["high"] < df["close"]) |
    (df["high"] < df["low"])
)

bad_low = (
    (df["low"] > df["open"]) |
    (df["low"] > df["close"]) |
    (df["low"] > df["high"])
)

print("bad high rows:", bad_high.sum())
print("bad low rows: ", bad_low.sum())

print("non-positive price:", (df[["open", "high", "low", "close"]] <= 0).any(axis=1).sum())
print("negative volume:", (df["volume"] < 0).sum())

bad high rows: 0
bad low rows:  0
non-positive price: 0
negative volume: 0


In [5]:
diff = df["date"].diff()

gap_mask = diff > pd.Timedelta(minutes=5)

gaps = df.loc[gap_mask, ["date"]].copy()
gaps["previous_date"] = df["date"].shift(1)[gap_mask]
gaps["gap"] = diff[gap_mask]

print(gaps)

                            date             previous_date             gap
23768  2023-03-24 14:00:00+00:00 2023-03-24 12:35:00+00:00 0 days 01:25:00
198354 2024-11-20 00:00:00+00:00 2024-11-19 18:45:00+00:00 0 days 05:15:00


In [6]:
gap_indices = df.index[gap_mask]

for idx in gap_indices:
    print("\n--- GAP ---")
    print(df.loc[max(0, idx - 3): idx + 3, ["date", "open", "close"]])


--- GAP ---
                           date      open     close
23765 2023-03-24 12:25:00+00:00  28080.00  28080.00
23766 2023-03-24 12:30:00+00:00  28080.00  28080.00
23767 2023-03-24 12:35:00+00:00  28080.00  28080.00
23768 2023-03-24 14:00:00+00:00  28079.99  27858.24
23769 2023-03-24 14:05:00+00:00  27858.23  27916.47
23770 2023-03-24 14:10:00+00:00  27916.47  28160.01
23771 2023-03-24 14:15:00+00:00  28160.01  28090.25

--- GAP ---
                            date      open     close
198351 2024-11-19 18:35:00+00:00  93610.99  93700.00
198352 2024-11-19 18:40:00+00:00  93700.00  93648.48
198353 2024-11-19 18:45:00+00:00  93648.47  93523.22
198354 2024-11-20 00:00:00+00:00  92310.80  92212.00
198355 2024-11-20 00:05:00+00:00  92211.99  92379.67
198356 2024-11-20 00:10:00+00:00  92379.67  92179.50
198357 2024-11-20 00:15:00+00:00  92179.50  92108.01


# Divide data into segments

In [7]:
expected = pd.Timedelta(minutes=5)

df["is_gap"] = df["date"].diff().ne(expected)

# 第一行不是 gap
df.loc[df.index[0], "is_gap"] = False

# 每遇到一个 gap，就进入新的连续数据段
df["segment_id"] = df["is_gap"].cumsum()

print(df["segment_id"].value_counts().sort_index())

segment_id
0     23768
1    174586
2     38084
Name: count, dtype: int64


# signal construction

## EMA20

In [9]:
df["ema20"] = (
    df.groupby("segment_id")["close"]
      .transform(lambda x: x.ewm(span=20, adjust=False).mean())
)

In [10]:
print(
    df[["date", "close", "ema20", "segment_id"]]
    .head(25)
)

                        date     close         ema20  segment_id
0  2023-01-01 00:00:00+00:00  16535.38  16535.380000           0
1  2023-01-01 00:05:00+00:00  16526.67  16534.550476           0
2  2023-01-01 00:10:00+00:00  16520.69  16533.230431           0
3  2023-01-01 00:15:00+00:00  16534.94  16533.393247           0
4  2023-01-01 00:20:00+00:00  16535.54  16533.597700           0
5  2023-01-01 00:25:00+00:00  16544.19  16534.606490           0
6  2023-01-01 00:30:00+00:00  16527.83  16533.961110           0
7  2023-01-01 00:35:00+00:00  16524.04  16533.016242           0
8  2023-01-01 00:40:00+00:00  16515.43  16531.341362           0
9  2023-01-01 00:45:00+00:00  16529.16  16531.133613           0
10 2023-01-01 00:50:00+00:00  16531.84  16531.200888           0
11 2023-01-01 00:55:00+00:00  16529.67  16531.055089           0
12 2023-01-01 01:00:00+00:00  16531.02  16531.051748           0
13 2023-01-01 01:05:00+00:00  16539.20  16531.827772           0
14 2023-01-01 01:10:00+00

In [11]:
df["bars_in_segment"] = df.groupby("segment_id").cumcount() + 1

df["ema_ready"] = df["bars_in_segment"] >= 20

# Assumptional Condition

In [13]:
df["above_ema"] = (
    (df["open"] > df["ema20"]) &
    (df["close"] > df["ema20"]) &
    df["ema_ready"]
)

In [14]:
print(
    df[
        ["date", "open", "close",
         "ema20", "above_ema",
         "segment_id", "bars_in_segment"]
    ].head(25)
)

                        date      open     close         ema20  above_ema  \
0  2023-01-01 00:00:00+00:00  16541.77  16535.38  16535.380000      False   
1  2023-01-01 00:05:00+00:00  16534.91  16526.67  16534.550476      False   
2  2023-01-01 00:10:00+00:00  16526.67  16520.69  16533.230431      False   
3  2023-01-01 00:15:00+00:00  16521.26  16534.94  16533.393247      False   
4  2023-01-01 00:20:00+00:00  16534.94  16535.54  16533.597700      False   
5  2023-01-01 00:25:00+00:00  16535.54  16544.19  16534.606490      False   
6  2023-01-01 00:30:00+00:00  16544.19  16527.83  16533.961110      False   
7  2023-01-01 00:35:00+00:00  16527.27  16524.04  16533.016242      False   
8  2023-01-01 00:40:00+00:00  16524.83  16515.43  16531.341362      False   
9  2023-01-01 00:45:00+00:00  16515.91  16529.16  16531.133613      False   
10 2023-01-01 00:50:00+00:00  16529.69  16531.84  16531.200888      False   
11 2023-01-01 00:55:00+00:00  16531.84  16529.67  16531.055089      False   

# Signal Construction

我们的定义是：

在 candle t 收盘后，如果截至 t 为止最近 20 根连续的 5m candles 都满足
open > EMA20 且 close > EMA20，则 signal[t] = True。

In [15]:
df["above_ema_20_count"] = (
    df.groupby("segment_id")["above_ema"]
      .transform(lambda x: x.rolling(window=20, min_periods=20).sum())
)

df["signal"] = df["above_ema_20_count"].eq(20)

## sanity checks

In [16]:
print("Total signals:", df["signal"].sum())

print(
    df.loc[df["signal"],
           ["date", "open", "close", "ema20",
            "above_ema_20_count", "signal"]]
      .head(10)
)

Total signals: 14397
                         date      open     close         ema20  \
134 2023-01-01 11:10:00+00:00  16549.16  16547.92  16539.120801   
135 2023-01-01 11:15:00+00:00  16547.56  16543.33  16539.521677   
136 2023-01-01 11:20:00+00:00  16543.31  16546.11  16540.149137   
137 2023-01-01 11:25:00+00:00  16546.11  16544.56  16540.569219   
138 2023-01-01 11:30:00+00:00  16544.56  16546.61  16541.144531   
139 2023-01-01 11:35:00+00:00  16546.25  16549.41  16541.931719   
140 2023-01-01 11:40:00+00:00  16549.41  16550.95  16542.790603   
141 2023-01-01 11:45:00+00:00  16550.59  16549.61  16543.440069   
142 2023-01-01 11:50:00+00:00  16549.61  16549.35  16544.002920   
143 2023-01-01 11:55:00+00:00  16549.10  16556.66  16545.208356   

     above_ema_20_count  signal  
134                20.0    True  
135                20.0    True  
136                20.0    True  
137                20.0    True  
138                20.0    True  
139                20.0    True  
140

In [17]:
first_signal_idx = df.index[df["signal"]][0]

print(
    df.loc[first_signal_idx - 21:first_signal_idx + 2,
           ["date", "open", "close", "ema20",
            "above_ema", "above_ema_20_count", "signal"]]
)

                         date      open     close         ema20  above_ema  \
113 2023-01-01 09:25:00+00:00  16520.20  16514.23  16517.903404      False   
114 2023-01-01 09:30:00+00:00  16514.64  16521.37  16518.233556      False   
115 2023-01-01 09:35:00+00:00  16521.30  16520.99  16518.496074       True   
116 2023-01-01 09:40:00+00:00  16520.99  16531.85  16519.767877       True   
117 2023-01-01 09:45:00+00:00  16531.85  16532.40  16520.970936       True   
118 2023-01-01 09:50:00+00:00  16532.40  16536.36  16522.436561       True   
119 2023-01-01 09:55:00+00:00  16536.36  16537.88  16523.907365       True   
120 2023-01-01 10:00:00+00:00  16538.32  16535.00  16524.963806       True   
121 2023-01-01 10:05:00+00:00  16535.37  16534.36  16525.858682       True   
122 2023-01-01 10:10:00+00:00  16534.86  16542.85  16527.476903       True   
123 2023-01-01 10:15:00+00:00  16542.60  16541.94  16528.854341       True   
124 2023-01-01 10:20:00+00:00  16541.94  16545.38  16530.428213 

In [18]:
df["entry_time"] = (
    df.groupby("segment_id")["date"]
      .shift(-1)
)

df["entry_price"] = (
    df.groupby("segment_id")["open"]
      .shift(-1)
)

df["exit_time"] = (
    df.groupby("segment_id")["date"]
      .shift(-13)
)

df["exit_price"] = (
    df.groupby("segment_id")["open"]
      .shift(-13)
)

In [19]:
df["forward_return_1h"] = (
    df["exit_price"] / df["entry_price"] - 1
)

## timing sanity check

In [20]:
df["holding_period"] = (
    df["exit_time"] - df["entry_time"]
)

print(df["holding_period"].value_counts(dropna=False).head())

holding_period
0 days 01:00:00    236399
NaT                    39
Name: count, dtype: int64


In [21]:
bad_horizon = (
    df["holding_period"].notna()
    & (df["holding_period"] != pd.Timedelta(hours=1))
)

print("Bad 1h horizons:", bad_horizon.sum())

Bad 1h horizons: 0


## 检查第一个signal

In [22]:
cols = [
    "date",
    "signal",
    "entry_time",
    "entry_price",
    "exit_time",
    "exit_price",
    "forward_return_1h",
]

print(df.loc[134, cols])

date                 2023-01-01 11:10:00+00:00
signal                                    True
entry_time           2023-01-01 11:15:00+00:00
entry_price                           16547.56
exit_time            2023-01-01 12:15:00+00:00
exit_price                            16564.37
forward_return_1h                     0.001016
Name: 134, dtype: object


In [23]:
print(
    df.loc[134:147,
           ["date", "open", "close"]]
)

                         date      open     close
134 2023-01-01 11:10:00+00:00  16549.16  16547.92
135 2023-01-01 11:15:00+00:00  16547.56  16543.33
136 2023-01-01 11:20:00+00:00  16543.31  16546.11
137 2023-01-01 11:25:00+00:00  16546.11  16544.56
138 2023-01-01 11:30:00+00:00  16544.56  16546.61
139 2023-01-01 11:35:00+00:00  16546.25  16549.41
140 2023-01-01 11:40:00+00:00  16549.41  16550.95
141 2023-01-01 11:45:00+00:00  16550.59  16549.61
142 2023-01-01 11:50:00+00:00  16549.61  16549.35
143 2023-01-01 11:55:00+00:00  16549.10  16556.66
144 2023-01-01 12:00:00+00:00  16556.66  16562.56
145 2023-01-01 12:05:00+00:00  16562.56  16551.88
146 2023-01-01 12:10:00+00:00  16551.88  16564.36
147 2023-01-01 12:15:00+00:00  16564.37  16561.28


In [24]:
manual_return = (
    df.loc[147, "open"] /
    df.loc[135, "open"]
    - 1
)

print(manual_return)
print(df.loc[134, "forward_return_1h"])

0.0010158597400460323
0.0010158597400460323


## E[r1h​∣Signal] vs E[r1h​]

### 先建立可用于分析的样本

In [26]:
valid = df["forward_return_1h"].notna()

signal_returns = df.loc[
    valid & df["signal"],
    "forward_return_1h"
]

all_returns = df.loc[
    valid,
    "forward_return_1h"
]

### 计算最基础的 descriptive statistics

In [27]:
results = {
    "signal_count": len(signal_returns),

    "conditional_mean": signal_returns.mean(),
    "unconditional_mean": all_returns.mean(),

    "difference": signal_returns.mean() - all_returns.mean(),

    "conditional_median": signal_returns.median(),
    "unconditional_median": all_returns.median(),

    "conditional_win_rate": (signal_returns > 0).mean(),
    "unconditional_win_rate": (all_returns > 0).mean(),
}

for k, v in results.items():
    print(f"{k}: {v}")

signal_count: 14396
conditional_mean: -4.870238958599862e-05
unconditional_mean: 9.603247715763586e-05
difference: -0.00014473486674363448
conditional_median: -0.0003986669797718667
unconditional_median: 5.267858720792162e-05
conditional_win_rate: 0.4449847179772159
unconditional_win_rate: 0.5082085795625193


In [29]:
print(
    df.loc[
        df["signal"] & df["forward_return_1h"].isna(),
        ["date", "segment_id", "signal"]
    ]
)

                            date  segment_id  signal
198341 2024-11-19 17:45:00+00:00           1    True


很可能有一个 signal 位于某个 continuous segment 尾部，没有完整未来 1h 数据

### 再把 return 转成 basis points (bps) 看，会更直观

1 bp=0.01%=0.0001

In [28]:
print(
    "Conditional mean (bps):",
    signal_returns.mean() * 10_000
)

print(
    "Unconditional mean (bps):",
    all_returns.mean() * 10_000
)

print(
    "Difference (bps):",
    (signal_returns.mean() - all_returns.mean()) * 10_000
)

Conditional mean (bps): -0.48702389585998623
Unconditional mean (bps): 0.9603247715763586
Difference (bps): -1.4473486674363447


# Result

在 2023-01-01 到 2025-04-01 的 BTC/USDT 5m 数据上，当连续 20 根 candle 的 Open 和 Close 均位于 EMA20 上方后，随后从 t+1 open 开始的 1h return，反而低于 unconditional 1h forward return。

# 下一步：先尝试证明这个结果是不是假的

## year breakdown

我们不调参数。

先问：

-1.447 bps 是否只是某一个特殊 market regime 拉出来的？

做最简单的 year breakdown。

In [31]:
df["year"] = df["date"].dt.year

yearly_results = []

for year, g in df.groupby("year"):

    valid = g["forward_return_1h"].notna()

    signal_r = g.loc[
        valid & g["signal"],
        "forward_return_1h"
    ]

    all_r = g.loc[
        valid,
        "forward_return_1h"
    ]

    yearly_results.append({
        "year": year,
        "signal_count": len(signal_r),
        "signal_mean_bps": signal_r.mean() * 10_000,
        "all_mean_bps": all_r.mean() * 10_000,
        "difference_bps": (
            signal_r.mean() - all_r.mean()
        ) * 10_000,
        "signal_win_rate": (signal_r > 0).mean(),
    })

yearly_results = pd.DataFrame(yearly_results)

print(yearly_results)

   year  signal_count  signal_mean_bps  all_mean_bps  difference_bps  \
0  2023          5892         2.266293      1.178505        1.087789   
1  2024          6815        -2.289557      1.074415       -3.363972   
2  2025          1689        -2.818746     -0.385053       -2.433694   

   signal_win_rate  
0         0.440597  
1         0.438004  
2         0.488455  


## Evidence：EXP-001 的 conditional return 存在明显 time instability / possible regime dependence。2023 为正，2024 与 2025 YTD 为负

# 构造 first_trigger

In [32]:
previous_signal = (
    df.groupby("segment_id")["signal"]
      .shift(1)
      .fillna(False)
      .astype(bool)
)

df["first_trigger"] = (
    df["signal"] & ~previous_signal
)

print("All signal observations:", df["signal"].sum())
print("First trigger events:", df["first_trigger"].sum())

All signal observations: 14397
First trigger events: 1368


/tmp/ipykernel_2411/2527445633.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [33]:
valid_first = (
    df["first_trigger"]
    & df["forward_return_1h"].notna()
)

first_returns = df.loc[
    valid_first,
    "forward_return_1h"
]

print("First trigger count:", len(first_returns))
print("Mean (bps):", first_returns.mean() * 10_000)
print("Median (bps):", first_returns.median() * 10_000)
print("Win rate:", (first_returns > 0).mean())

First trigger count: 1368
Mean (bps): 3.4635985196284182
Median (bps): -1.4799476891802499
Win rate: 0.4766081871345029


In [34]:
print(
    "Difference vs unconditional (bps):",
    (first_returns.mean() - all_returns.mean()) * 10_000
)

Difference vs unconditional (bps): 2.50327374805206


## sanity check 看看 first triggers 实际间隔多远：

In [35]:
first_trigger_times = df.loc[
    df["first_trigger"],
    "date"
]

print(first_trigger_times.diff().describe())
print("Minimum gap:", first_trigger_times.diff().min())

count                         1367
mean     0 days 14:24:30.153621068
std      0 days 11:54:05.723123167
min                0 days 01:45:00
25%                0 days 06:05:00
50%                0 days 11:05:00
75%                0 days 19:02:30
max                4 days 23:50:00
Name: date, dtype: object
Minimum gap: 0 days 01:45:00


## 最后，再按年份拆 first trigger

In [36]:
first_yearly = []

for year, g in df.groupby("year"):

    valid = (
        g["first_trigger"]
        & g["forward_return_1h"].notna()
    )

    r = g.loc[valid, "forward_return_1h"]

    first_yearly.append({
        "year": year,
        "event_count": len(r),
        "mean_bps": r.mean() * 10_000,
        "median_bps": r.median() * 10_000,
        "win_rate": (r > 0).mean(),
    })

print(pd.DataFrame(first_yearly))

   year  event_count  mean_bps  median_bps  win_rate
0  2023          567  5.288293   -0.850838  0.483245
1  2024          643  2.437059   -3.219377  0.468118
2  2025          158  1.093111   -0.961365  0.487342


# 下一步：检查 return distribution

In [37]:
r = first_returns

quantiles = (
    r.quantile([
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ])
    * 10_000
)

print("Return quantiles (bps):")
print(quantiles)

Return quantiles (bps):
0.01   -128.676333
0.05    -66.660295
0.10    -46.236840
0.25    -21.222990
0.50     -1.479948
0.75     19.733574
0.90     56.624696
0.95    102.493184
0.99    202.481568
Name: forward_return_1h, dtype: float64


## 然后分开看 winners / losers

In [38]:
winners = r[r > 0]
losers = r[r < 0]

print("Win probability:", len(winners) / len(r))

print(
    "Average winner (bps):",
    winners.mean() * 10_000
)

print(
    "Average loser (bps):",
    losers.mean() * 10_000
)

print(
    "Best event (bps):",
    r.max() * 10_000
)

print(
    "Worst event (bps):",
    r.min() * 10_000
)

Win probability: 0.4766081871345029
Average winner (bps): 40.33432272465257
Average loser (bps): -30.11141849388519
Best event (bps): 465.92459259077845
Worst event (bps): -337.28310029855993


## 理解 Expected Value：

EV=P(win)×E[win]+P(loss)×E[loss]

In [39]:
p_win = (r > 0).mean()
p_loss = (r < 0).mean()

ev = (
    p_win * winners.mean()
    +
    p_loss * losers.mean()
)

print("EV from decomposition (bps):", ev * 10_000)
print("Actual mean (bps):", r.mean() * 10_000)

EV from decomposition (bps): 3.4635985196284214
Actual mean (bps): 3.4635985196284182


## 然后做一个非常重要的 outlier stress test

In [40]:
q99 = r.quantile(0.99)
q95 = r.quantile(0.95)

without_top_1pct = r[r <= q99]
without_top_5pct = r[r <= q95]

print("Original mean (bps):",
      r.mean() * 10_000)

print("Without top 1% winners (bps):",
      without_top_1pct.mean() * 10_000)

print("Without top 5% winners (bps):",
      without_top_5pct.mean() * 10_000)

Original mean (bps): 3.4635985196284182
Without top 1% winners (bps): 0.590025968342629
Without top 5% winners (bps): -5.110733632435361


# 一个更公平的 distribution robustness test

In [41]:
def symmetric_trimmed_mean(r, trim):
    lower = r.quantile(trim)
    upper = r.quantile(1 - trim)

    trimmed = r[
        (r >= lower) &
        (r <= upper)
    ]

    return {
        "n": len(trimmed),
        "mean_bps": trimmed.mean() * 10_000,
        "median_bps": trimmed.median() * 10_000,
        "win_rate": (trimmed > 0).mean(),
    }


print("1% symmetric trim:")
print(symmetric_trimmed_mean(first_returns, 0.01))

print("\n5% symmetric trim:")
print(symmetric_trimmed_mean(first_returns, 0.05))

1% symmetric trim:
{'n': 1340, 'mean_bps': 2.6370559472264152, 'median_bps': -1.4799476891802499, 'win_rate': 0.4761194029850746}

5% symmetric trim:
{'n': 1230, 'mean_bps': 0.7573072537301075, 'median_bps': -1.4799476891802499, 'win_rate': 0.4739837398373984}


# Monthly Robustness

In [42]:
df["month"] = df["date"].dt.to_period("M")

monthly_results = []

for month, g in df.groupby("month"):

    valid = (
        g["first_trigger"]
        & g["forward_return_1h"].notna()
    )

    r = g.loc[valid, "forward_return_1h"]

    if len(r) == 0:
        continue

    monthly_results.append({
        "month": str(month),
        "event_count": len(r),
        "mean_bps": r.mean() * 10_000,
        "median_bps": r.median() * 10_000,
        "win_rate": (r > 0).mean(),
    })

monthly_results = pd.DataFrame(monthly_results)

print(monthly_results)

      month  event_count   mean_bps  median_bps  win_rate
0   2023-01           52  16.681904    2.395433  0.576923
1   2023-02           41   0.640971   -4.247414  0.365854
2   2023-03           56  13.778185   -1.574438  0.464286
3   2023-04           36   4.008787   -2.240762  0.472222
4   2023-05           45   1.180012    0.970616  0.577778
5   2023-06           46  -1.190279   -8.162676  0.369565
6   2023-07           39  -2.077042   -1.866021  0.384615
7   2023-08           35  14.605946   -2.110277  0.371429
8   2023-09           43   8.456805   -3.354643  0.488372
9   2023-10           65   3.404157    1.332641  0.538462
10  2023-11           55  -8.239057   -4.134634  0.454545
11  2023-12           54  11.639128    9.266860  0.629630
12  2024-01           52   8.597488    3.551857  0.576923
13  2024-02           59  10.629845   -0.437965  0.491525
14  2024-03           58  -2.814602   -0.849728  0.482759
15  2024-04           51   0.639154  -10.323842  0.372549
16  2024-05   

/tmp/ipykernel_2411/633177466.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["date"].dt.to_period("M")


## 汇总

In [43]:
print(
    "Positive mean months:",
    (monthly_results["mean_bps"] > 0).sum()
)

print(
    "Total months:",
    len(monthly_results)
)

print(
    "Pct positive months:",
    (monthly_results["mean_bps"] > 0).mean()
)

print(
    "Median monthly mean (bps):",
    monthly_results["mean_bps"].median()
)

Positive mean months: 18
Total months: 28
Pct positive months: 0.6428571428571429
Median monthly mean (bps): 2.292084285655557
